# Market Microstructure Analysis

## Deep Dive into US Interest Rate Swap Market Structure

This notebook provides comprehensive market microstructure analysis:

1. **Trade Size Distribution** - Notional size analysis and percentiles
2. **Block Trade Analysis** - Block threshold analysis and characteristics
3. **SEF vs Off-Facility** - Venue competition and market share
4. **Execution Latency** - Time-to-dissemination analysis
5. **Package Trades** - Multi-leg trade structures
6. **Liquidity Metrics** - Trade frequency and market depth indicators

In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "vscode"

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
from scipy import stats

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York")
CHI_tz = pytz.timezone("America/Chicago")
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
from MDP.IRSwaps.SDR_INTRADAY.rl_curve_utils.SDRDataBuilder import SDRDataBuilder

cache_path = r"/tmp/sdr_cache"
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)

## 1. Data Loading and Preprocessing

In [ ]:
# Fetch multiple days for more robust microstructure analysis
start = NY_tz.localize(datetime.datetime(2025, 12, 16, 7, 0))
end = NY_tz.localize(datetime.datetime(2025, 12, 19, 17, 0))

raw_df = sdr.grab_sdr_trades(
    start_timestamp=start,
    end_timestamp=end,
    agency="CFTC",
    asset_class="RATES"
)

print(f"Total trades fetched: {len(raw_df):,}")

In [ ]:
def preprocess_microstructure_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Preprocess SDR data for microstructure analysis.
    """
    df = df.copy()
    
    # Filter to new trades
    df = df[df['Action type'] == 'NEWT'].copy()
    
    # Parse notional amounts
    for col in ['Notional amount-Leg 1', 'Notional amount-Leg 2']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(',', '').str.replace(' ', '')
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Parse timestamps
    df['Event timestamp'] = pd.to_datetime(df['Event timestamp'], utc=True)
    df['Execution Timestamp'] = pd.to_datetime(df['Execution Timestamp'], utc=True)
    df['Event_Time_NY'] = df['Event timestamp'].dt.tz_convert('America/New_York')
    df['Execution_Time_NY'] = df['Execution Timestamp'].dt.tz_convert('America/New_York')
    
    # Calculate dissemination latency (seconds)
    df['Dissemination_Latency_Sec'] = (df['Event timestamp'] - df['Execution Timestamp']).dt.total_seconds()
    # Filter out obviously bad latencies
    df.loc[df['Dissemination_Latency_Sec'] < 0, 'Dissemination_Latency_Sec'] = np.nan
    df.loc[df['Dissemination_Latency_Sec'] > 86400, 'Dissemination_Latency_Sec'] = np.nan  # > 1 day
    
    # Parse dates for tenor
    df['Effective Date'] = pd.to_datetime(df['Effective Date'], errors='coerce')
    df['Expiration Date'] = pd.to_datetime(df['Expiration Date'], errors='coerce')
    df['Tenor_Days'] = (df['Expiration Date'] - df['Effective Date']).dt.days
    df['Tenor_Years'] = df['Tenor_Days'] / 365.25
    
    # Venue classification
    def classify_venue(platform):
        platform = str(platform).upper()
        if platform in ['XOFF', 'OFF', 'NaN', '', 'NONE', 'NAN']:
            return 'Off-Facility'
        else:
            return 'SEF'
    df['Venue_Type'] = df['Platform identifier'].apply(classify_venue)
    
    # Block trade indicator
    df['Is_Block'] = df['Block trade election indicator'].astype(str).str.upper() == 'TRUE'
    
    # Package indicator
    df['Is_Package'] = df['Package indicator'].astype(str).str.upper() == 'TRUE'
    
    # Product classification
    def classify_product(row):
        fisn = str(row.get('UPI FISN', '')).upper()
        underlier = str(row.get('UPI Underlier Name', '')).upper()
        
        if 'OIS' in fisn or 'COMPOUND' in underlier:
            return 'OIS'
        elif 'FXD FLT' in fisn or 'FIXED' in fisn:
            return 'Fixed-Float'
        elif 'BASIS' in fisn:
            return 'Basis'
        elif 'SWAPTION' in fisn or 'CALL' in fisn or 'PUT' in fisn:
            return 'Swaption'
        elif 'CAP' in fisn or 'FLOOR' in fisn:
            return 'Cap/Floor'
        else:
            return 'Other'
    df['Product_Type'] = df.apply(classify_product, axis=1)
    
    # Filter USD
    df = df[df['Notional currency-Leg 1'] == 'USD'].copy()
    
    # Filter SOFR
    sofr_mask = df['UPI Underlier Name'].str.contains('SOFR', case=False, na=False)
    df = df[sofr_mask].copy()
    
    return df.reset_index(drop=True)

df = preprocess_microstructure_data(raw_df)
print(f"Processed USD SOFR trades: {len(df):,}")

## 2. Trade Size Distribution Analysis

In [ ]:
# Overall notional distribution
notional = df['Notional amount-Leg 1'].dropna()

print("Trade Size Statistics (USD SOFR):")
print("="*50)
print(f"Count: {len(notional):,}")
print(f"Mean: ${notional.mean():,.0f}")
print(f"Median: ${notional.median():,.0f}")
print(f"Std Dev: ${notional.std():,.0f}")
print(f"Min: ${notional.min():,.0f}")
print(f"Max: ${notional.max():,.0f}")
print(f"\nPercentiles:")
for p in [5, 10, 25, 50, 75, 90, 95, 99]:
    print(f"  P{p}: ${notional.quantile(p/100):,.0f}")

In [ ]:
# Histogram of trade sizes (log scale)
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=np.log10(notional.clip(lower=1)),
    nbinsx=50,
    marker_color='steelblue',
    opacity=0.75
))

# Add percentile lines
for p, color in [(50, 'green'), (90, 'orange'), (99, 'red')]:
    val = np.log10(notional.quantile(p/100))
    fig.add_vline(x=val, line_dash='dash', line_color=color,
                  annotation_text=f'P{p}', annotation_position='top right')

fig.update_layout(
    title='Trade Size Distribution (Log Scale)',
    xaxis_title='Log10(Notional)',
    yaxis_title='Count',
    height=500
)

# Add custom x-axis labels
tickvals = [5, 6, 7, 8, 9, 10]
ticktext = ['$100K', '$1M', '$10M', '$100M', '$1B', '$10B']
fig.update_xaxes(tickvals=tickvals, ticktext=ticktext)

fig.show()

In [ ]:
# Trade size distribution by venue
fig = go.Figure()

for venue in ['SEF', 'Off-Facility']:
    venue_data = df[df['Venue_Type'] == venue]['Notional amount-Leg 1'].dropna()
    fig.add_trace(go.Histogram(
        x=np.log10(venue_data.clip(lower=1)),
        name=venue,
        opacity=0.6,
        nbinsx=40
    ))

fig.update_layout(
    title='Trade Size Distribution by Venue',
    xaxis_title='Log10(Notional)',
    yaxis_title='Count',
    barmode='overlay',
    height=500
)
fig.update_xaxes(tickvals=tickvals, ticktext=ticktext)
fig.show()

In [ ]:
# Trade size by product type
size_by_product = df.groupby('Product_Type')['Notional amount-Leg 1'].agg(['count', 'mean', 'median', 'sum']).round(0)
size_by_product.columns = ['Trade_Count', 'Avg_Size', 'Median_Size', 'Total_Notional']
size_by_product = size_by_product.sort_values('Trade_Count', ascending=False)
size_by_product

## 3. Block Trade Analysis

In [ ]:
# Block vs non-block statistics
block_stats = df.groupby('Is_Block').agg({
    'Dissemination Identifier': 'count',
    'Notional amount-Leg 1': ['sum', 'mean', 'median']
}).round(0)
block_stats.columns = ['Trade_Count', 'Total_Notional', 'Avg_Notional', 'Median_Notional']
block_stats.index = block_stats.index.map({True: 'Block', False: 'Non-Block'})

print("Block Trade Statistics:")
display(block_stats)

In [ ]:
# Block trade percentage by product
block_by_product = df.groupby(['Product_Type', 'Is_Block']).size().unstack(fill_value=0)
block_by_product.columns = ['Non-Block', 'Block']
block_by_product['Block_Pct'] = block_by_product['Block'] / (block_by_product['Block'] + block_by_product['Non-Block']) * 100
block_by_product = block_by_product.sort_values('Block_Pct', ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=block_by_product.index,
    y=block_by_product['Block_Pct'],
    marker_color='darkred',
    text=[f"{v:.1f}%" for v in block_by_product['Block_Pct']],
    textposition='auto'
))

fig.update_layout(
    title='Block Trade Percentage by Product Type',
    xaxis_title='Product Type',
    yaxis_title='Block Trade %',
    height=400
)
fig.show()

In [ ]:
# Block trade size thresholds by tenor
# CFTC block thresholds vary by tenor - let's analyze where actual block trades fall

def assign_tenor_bucket(years):
    if pd.isna(years) or years <= 0:
        return 'Unknown'
    elif years <= 2:
        return '0-2Y'
    elif years <= 5:
        return '2-5Y'
    elif years <= 10:
        return '5-10Y'
    elif years <= 30:
        return '10-30Y'
    else:
        return '30Y+'

df['Tenor_Bucket_Block'] = df['Tenor_Years'].apply(assign_tenor_bucket)

block_trades = df[df['Is_Block']].copy()
if len(block_trades) > 0:
    block_size_by_tenor = block_trades.groupby('Tenor_Bucket_Block')['Notional amount-Leg 1'].agg(['count', 'min', 'median', 'mean', 'max'])
    block_size_by_tenor.columns = ['Count', 'Min', 'Median', 'Mean', 'Max']
    print("Block Trade Size Statistics by Tenor:")
    display(block_size_by_tenor.round(0))
else:
    print("No block trades in dataset")

## 4. Venue Competition Analysis

In [ ]:
# SEF market share
sef_trades = df[df['Venue_Type'] == 'SEF']
platform_share = sef_trades.groupby('Platform identifier').agg({
    'Dissemination Identifier': 'count',
    'Notional amount-Leg 1': 'sum'
}).reset_index()
platform_share.columns = ['Platform', 'Trade_Count', 'Total_Notional']
platform_share['Trade_Share_Pct'] = platform_share['Trade_Count'] / platform_share['Trade_Count'].sum() * 100
platform_share['Notional_Share_Pct'] = platform_share['Total_Notional'] / platform_share['Total_Notional'].sum() * 100
platform_share = platform_share.sort_values('Trade_Count', ascending=False)

print("Top 10 SEFs by Trade Count:")
display(platform_share.head(10))

In [ ]:
# SEF market share visualization
top_sefs = platform_share.head(8)
other_count = platform_share.iloc[8:]['Trade_Count'].sum() if len(platform_share) > 8 else 0
other_notional = platform_share.iloc[8:]['Total_Notional'].sum() if len(platform_share) > 8 else 0

if other_count > 0:
    top_sefs = pd.concat([top_sefs, pd.DataFrame([{'Platform': 'Other', 'Trade_Count': other_count, 'Total_Notional': other_notional}])])

fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'pie'}, {'type': 'pie'}]],
                    subplot_titles=['Trade Count Share', 'Notional Share'])

fig.add_trace(
    go.Pie(labels=top_sefs['Platform'], values=top_sefs['Trade_Count'],
           textinfo='label+percent', hole=0.3),
    row=1, col=1
)

fig.add_trace(
    go.Pie(labels=top_sefs['Platform'], values=top_sefs['Total_Notional'],
           textinfo='label+percent', hole=0.3),
    row=1, col=2
)

fig.update_layout(title_text='SEF Market Share', height=500)
fig.show()

In [ ]:
# SEF vs Off-Facility by tenor
venue_by_tenor = df.groupby(['Tenor_Bucket_Block', 'Venue_Type']).size().unstack(fill_value=0)
venue_by_tenor['Total'] = venue_by_tenor.sum(axis=1)
venue_by_tenor['SEF_Pct'] = venue_by_tenor.get('SEF', 0) / venue_by_tenor['Total'] * 100

tenor_order = ['0-2Y', '2-5Y', '5-10Y', '10-30Y', '30Y+', 'Unknown']
venue_by_tenor = venue_by_tenor.reindex([t for t in tenor_order if t in venue_by_tenor.index])

fig = go.Figure()
fig.add_trace(go.Bar(
    x=venue_by_tenor.index,
    y=venue_by_tenor['SEF_Pct'],
    marker_color='steelblue',
    text=[f"{v:.1f}%" for v in venue_by_tenor['SEF_Pct']],
    textposition='auto'
))

fig.update_layout(
    title='SEF Trading Percentage by Tenor',
    xaxis_title='Tenor Bucket',
    yaxis_title='SEF %',
    height=400
)
fig.show()

## 5. Execution Latency Analysis

In [ ]:
# Dissemination latency statistics
latency = df['Dissemination_Latency_Sec'].dropna()

if len(latency) > 0:
    print("Dissemination Latency Statistics (seconds):")
    print("="*50)
    print(f"Count: {len(latency):,}")
    print(f"Mean: {latency.mean():.1f}s")
    print(f"Median: {latency.median():.1f}s")
    print(f"Std Dev: {latency.std():.1f}s")
    print(f"Min: {latency.min():.1f}s")
    print(f"Max: {latency.max():.1f}s")
    print(f"\nPercentiles:")
    for p in [50, 75, 90, 95, 99]:
        print(f"  P{p}: {latency.quantile(p/100):.1f}s")

In [ ]:
# Latency distribution
if len(latency) > 0:
    # Cap at 15 minutes for visualization
    latency_capped = latency.clip(upper=900)
    
    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=latency_capped,
        nbinsx=50,
        marker_color='steelblue',
        opacity=0.75
    ))
    
    fig.update_layout(
        title='Dissemination Latency Distribution (capped at 15 min)',
        xaxis_title='Latency (seconds)',
        yaxis_title='Count',
        height=400
    )
    fig.show()

In [ ]:
# Latency by venue
latency_by_venue = df.groupby('Venue_Type')['Dissemination_Latency_Sec'].agg(['count', 'mean', 'median'])
latency_by_venue.columns = ['Count', 'Mean_Latency', 'Median_Latency']
print("Latency by Venue Type:")
display(latency_by_venue.round(1))

In [ ]:
# Latency by block status
latency_by_block = df.groupby('Is_Block')['Dissemination_Latency_Sec'].agg(['count', 'mean', 'median'])
latency_by_block.columns = ['Count', 'Mean_Latency', 'Median_Latency']
latency_by_block.index = latency_by_block.index.map({True: 'Block', False: 'Non-Block'})
print("Latency by Block Status:")
display(latency_by_block.round(1))

## 6. Package Trade Analysis

In [ ]:
# Package trade statistics
package_stats = df.groupby('Is_Package').agg({
    'Dissemination Identifier': 'count',
    'Notional amount-Leg 1': ['sum', 'mean']
}).round(0)
package_stats.columns = ['Trade_Count', 'Total_Notional', 'Avg_Notional']
package_stats.index = package_stats.index.map({True: 'Package', False: 'Single-Leg'})

print("Package Trade Statistics:")
display(package_stats)

In [ ]:
# Package trades by product
package_by_product = df.groupby(['Product_Type', 'Is_Package']).size().unstack(fill_value=0)
package_by_product.columns = ['Single-Leg', 'Package']
package_by_product['Package_Pct'] = package_by_product['Package'] / (package_by_product['Package'] + package_by_product['Single-Leg']) * 100
package_by_product = package_by_product.sort_values('Package_Pct', ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=package_by_product.index,
    y=package_by_product['Package_Pct'],
    marker_color='purple',
    text=[f"{v:.1f}%" for v in package_by_product['Package_Pct']],
    textposition='auto'
))

fig.update_layout(
    title='Package Trade Percentage by Product Type',
    xaxis_title='Product Type',
    yaxis_title='Package %',
    height=400
)
fig.show()

## 7. Liquidity Metrics

In [ ]:
# Trades per hour
df['Hour_NY'] = df['Event_Time_NY'].dt.hour
df['Date'] = df['Event_Time_NY'].dt.date

trades_per_hour = df.groupby(['Date', 'Hour_NY']).size().reset_index(name='Trade_Count')
avg_trades_per_hour = trades_per_hour.groupby('Hour_NY')['Trade_Count'].mean()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=avg_trades_per_hour.index,
    y=avg_trades_per_hour.values,
    marker_color='steelblue'
))

fig.update_layout(
    title='Average Trades per Hour (NY Time)',
    xaxis_title='Hour (NY Time)',
    yaxis_title='Average Trade Count',
    height=400
)
fig.show()

In [ ]:
# Inter-trade time analysis
df_sorted = df.sort_values('Event timestamp')
df_sorted['Inter_Trade_Time'] = df_sorted['Event timestamp'].diff().dt.total_seconds()

inter_trade = df_sorted['Inter_Trade_Time'].dropna()
inter_trade = inter_trade[(inter_trade > 0) & (inter_trade < 3600)]  # Filter reasonable values

if len(inter_trade) > 0:
    print("Inter-Trade Time Statistics (seconds):")
    print("="*50)
    print(f"Mean: {inter_trade.mean():.2f}s")
    print(f"Median: {inter_trade.median():.2f}s")
    print(f"Std Dev: {inter_trade.std():.2f}s")
    print(f"\nPercentiles:")
    for p in [25, 50, 75, 90]:
        print(f"  P{p}: {inter_trade.quantile(p/100):.2f}s")

In [ ]:
# Daily trading activity
daily_activity = df.groupby('Date').agg({
    'Dissemination Identifier': 'count',
    'Notional amount-Leg 1': 'sum'
}).reset_index()
daily_activity.columns = ['Date', 'Trade_Count', 'Total_Notional']

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['Daily Trade Count', 'Daily Notional ($B)'])

fig.add_trace(
    go.Bar(x=daily_activity['Date'], y=daily_activity['Trade_Count'],
           name='Trade Count', marker_color='steelblue'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=daily_activity['Date'], y=daily_activity['Total_Notional']/1e9,
           name='Notional ($B)', marker_color='darkgreen'),
    row=2, col=1
)

fig.update_layout(height=600, title_text='Daily Trading Activity')
fig.show()

## 8. Summary Report

In [ ]:
print("="*70)
print("MARKET MICROSTRUCTURE SUMMARY")
print("="*70)
print(f"\nAnalysis Period: {start.strftime('%Y-%m-%d')} to {end.strftime('%Y-%m-%d')}")
print(f"\nVolume Metrics:")
print(f"  Total Trades: {len(df):,}")
print(f"  Total Notional: ${df['Notional amount-Leg 1'].sum():,.0f}")
print(f"  Average Trade Size: ${df['Notional amount-Leg 1'].mean():,.0f}")
print(f"  Median Trade Size: ${df['Notional amount-Leg 1'].median():,.0f}")

print(f"\nVenue Distribution:")
sef_pct = len(df[df['Venue_Type'] == 'SEF']) / len(df) * 100
print(f"  SEF Trading: {sef_pct:.1f}%")
print(f"  Off-Facility: {100-sef_pct:.1f}%")

print(f"\nBlock Trade Activity:")
block_pct = df['Is_Block'].sum() / len(df) * 100
print(f"  Block Trades: {block_pct:.1f}%")
print(f"  Average Block Size: ${df[df['Is_Block']]['Notional amount-Leg 1'].mean():,.0f}" if df['Is_Block'].sum() > 0 else "  No block trades")

print(f"\nPackage Trades:")
package_pct = df['Is_Package'].sum() / len(df) * 100
print(f"  Package Trade %: {package_pct:.1f}%")

print(f"\nExecution Latency:")
print(f"  Median Latency: {df['Dissemination_Latency_Sec'].median():.1f}s")
print(f"  Mean Latency: {df['Dissemination_Latency_Sec'].mean():.1f}s")